# Exploratory Data Analysis

In [34]:
# load packages
import pandas as pd
import numpy as np
import glob

In [35]:
# get list of file paths
file_paths = glob.glob("./Data/raw/citi_costco*.CSV")

# loop through each file in the folder and essentially union all the csvs together into one dataframe
citi_costco_transactions = pd.concat([pd.read_csv(file) for file in file_paths], ignore_index=True)

citi_costco_transactions.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 98 entries, 0 to 97
Data columns (total 6 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   Status       98 non-null     object 
 1   Date         98 non-null     object 
 2   Description  98 non-null     object 
 3   Debit        95 non-null     float64
 4   Credit       3 non-null      float64
 5   Member Name  96 non-null     object 
dtypes: float64(2), object(4)
memory usage: 4.7+ KB


In [57]:
from dataclasses import dataclass
from pathlib import Path
from __future__ import annotations

@dataclass
class RawStatement:
    """Container for one raw CSV plus metadata about where it came from."""
    source_name: str          # e.g. "chase_credit_card" — matches config.yaml key
    file_path: Path
    df: pd.DataFrame

In [88]:
# Load the configuration
import yaml

def load_config(config_path: str = "./config.yaml") -> dict:
    """Load the source/column-mapping config."""
    # TODO: open config_path, yaml.safe_load(), return dict
    with open(config_path, "r") as f:
        config = yaml.safe_load(f)
    return config

config__ = load_config()
print(config__["sources"].items())

dict_items([('citi_costco_credit_card', {'file_pattern': 'citi_costco*.csv', 'column_map': {'month': None, 'date': 'Date', 'category': None, 'subcategory': None, 'amount': 'Debit', 'description': 'Description'}, 'amount_sign': 'positive_is_expense', 'date_format': '%m/%d/%Y'}), ('amex_gold_credit_card', {'file_pattern': 'amex_gold*.csv', 'column_map': {'month': None, 'date': 'Date', 'category': None, 'subcategory': None, 'amount': 'Debit', 'description': 'Description'}, 'amount_sign': 'positive_is_expense', 'date_format': '%m/%d/%Y'}), ('iq_credit_card', {'file_pattern': 'iq_credit_card*.csv', 'column_map': {'month': None, 'date': 'Date', 'category': None, 'subcategory': None, 'amount': 'Amount', 'description': 'Description'}, 'amount_sign': 'positive_is_expense', 'date_format': '%m/%d/%Y'}), ('iq_checking', {'file_pattern': 'iq_checking*.csv', 'column_map': {'month': None, 'date': 'Posting Date', 'category': None, 'subcategory': None, 'amount': 'Amount', 'description': 'Description'},

In [102]:

def find_files_for_source(raw_data_dir: Path, file_pattern: str) -> list[Path]:
    """Glob raw_data_dir for files matching this source's pattern."""
    # TODO: return sorted list of Path objects matching file_pattern
    case_insensitive_pattern = file_pattern.replace(".csv", ".[cC][sS][vV]")
    file_paths = glob.glob(f"{raw_data_dir}/{case_insensitive_pattern}")
    return sorted(Path(f) for f in file_paths)

root_data_dir = "./Data/raw"
citi_files = find_files_for_source(root_data_dir, "citi_costco*.CSV")
amex_files = find_files_for_source(root_data_dir, "amex_gold_credit_card*.CSV")
iq_files = find_files_for_source(root_data_dir, "iq_credit_card*.CSV")

print("Citi files:", citi_files)
print("Amex files:", amex_files)
print("IQ files:", iq_files)


Citi files: [PosixPath('Data/raw/citi_costco_2026-06-19.CSV'), PosixPath('Data/raw/citi_costco_2026-07-21.CSV')]
Amex files: []
IQ files: []


In [106]:
def extract_all(config: dict) -> list[RawStatement]:
    """
    Main entry point for this module.
    Loop through every source in config['sources'], find matching files,
    read each into a DataFrame, and return a list of RawStatement objects.
    """
    raw_statements: list[RawStatement] = []
    root_data_dir = "./Data/raw"

    # TODO:
    for source_name, source_cfg in config["sources"].items():
        files = find_files_for_source(root_data_dir, source_cfg["file_pattern"])
        for f in files:
            df = pd.concat([pd.read_csv(f) for f in files], ignore_index=True)
            raw_statements.append(RawStatement(source_name, f, df))

    return raw_statements

if __name__ == "__main__":
    # Quick manual test when running this file directly
    cfg = load_config()
    statements = extract_all(cfg)
    print(f"Extracted {len(statements)} raw statement(s).")
    print(statements[0].df.head())


Extracted 2 raw statement(s).
    Status        Date                             Description  Debit  \
0  Cleared  06/19/2026             ZWIFT, INC. 855-469-9438 CA  21.73   
1  Cleared  06/19/2026  AMAZON MKTPL*256Q84NO3 Amzn.com/billWA  64.19   
2  Cleared  06/19/2026      INTEREST CHARGED TO STANDARD PURCH  20.67   
3  Cleared  06/17/2026               ONLINE PAYMENT, THANK YOU    NaN   
4  Cleared  06/16/2026          TRADER JOE S #272 VANCOUVER WA  99.45   

    Credit Member Name  
0      NaN  AUSTIN SEO  
1      NaN  AUSTIN SEO  
2      NaN  AUSTIN SEO  
3 -2315.61  AUSTIN SEO  
4      NaN  AUSTIN SEO  


In [104]:
# source_name = "citi_costco_credit_card"
# source_cfg = "citi_costco*.CSV"
root_data_dir = "./Data/raw"
config_items = load_config()
# print(config_items["sources"])  # Print the first item in the config dictionary
# print(config_items["file_pattern"])

for source_name, source_cfg in config_items["sources"].items():
    # print(f"Source: {source_name}, File Pattern: {source_cfg['file_pattern']}")
    files = find_files_for_source(root_data_dir, source_cfg["file_pattern"])
    for f in files:
        print(f)
# find_files_for_source(root_data_dir, source_cfg)

Data/raw/citi_costco_2026-06-19.CSV
Data/raw/citi_costco_2026-07-21.CSV
